# TruthLens AI — FEVER Claim-Only Baseline Model

### Task Definition
**FEVER CLAIM-ONLY BASELINE:** `claim -> label (SUPPORTS, REFUTES, NOT_ENOUGH_INFO)`

> **IMPORTANT NOTICE:**  
> This baseline model operates strictly on statement text alone because Wikipedia evidence text is not provided in raw FEVER files. It establishes the linguistic prior baseline (measuring lexical cues and crowd annotator artifacts in synthetic claims) and serves as the benchmark against which full claim+evidence verification models will be compared.

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.insert(0, os.path.abspath("../src"))
from training_utils import LABEL2ID, ID2LABEL, set_seed, compute_verification_metrics, plot_and_save_confusion_matrix
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

set_seed(42)
print("Libraries loaded and seed set to 42.")

## 1. Load Processed FEVER Splits
Ingesting clean, deduplicated FEVER data (`121,287` train claims, `13,477` validation claims).

In [ ]:
train_df = pd.read_csv("../data/final/verification/fever_train.csv")
valid_df = pd.read_csv("../data/final/verification/fever_valid.csv")

print(f"Train claims: {len(train_df):,}")
print(f"Valid claims: {len(valid_df):,}")
print("\nTrain label distribution:")
print(train_df["label"].value_counts(normalize=True) * 100)

## 2. Feature Extraction: Sublinear TF-IDF (Unigrams + Bigrams)

In [ ]:
tfidf = TfidfVectorizer(
    max_features=25000,
    ngram_range=(1, 2),
    sublinear_tf=True,
    strip_accents="unicode"
)

X_train = tfidf.fit_transform(train_df["claim"].astype(str))
X_valid = tfidf.transform(valid_df["claim"].astype(str))

y_train = train_df["label"].map(LABEL2ID).values
y_valid = valid_df["label"].map(LABEL2ID).values

print(f"TF-IDF Matrix shape (Train): {X_train.shape}")
print(f"TF-IDF Matrix shape (Valid): {X_valid.shape}")

## 3. Train Classifier: Multinomial Logistic Regression

In [ ]:
clf = LogisticRegression(
    C=1.0,
    max_iter=1000,
    class_weight="balanced",
    solver="lbfgs",
    random_state=42,
)
clf.fit(X_train, y_train)
print("Model training complete.")

## 4. Evaluation & Detailed Metrics

In [ ]:
y_pred = clf.predict(X_valid)
metrics = compute_verification_metrics(y_valid, y_pred)

print(f"Accuracy : {metrics['accuracy']*100:.2f}%")
print(f"Macro F1 : {metrics['macro_f1']*100:.2f}%")
print(f"Weighted F1 : {metrics['weighted_f1']*100:.2f}%")

print("\nPer-Class Breakdown:")
for k, v in metrics["per_class"].items():
    print(f"  {k:<16}: Precision={v['precision']*100:.2f}%, Recall={v['recall']*100:.2f}%, F1={v['f1']*100:.2f}%")

## 5. Confusion Matrix Visualization

In [ ]:
plot_and_save_confusion_matrix(
    metrics["confusion_matrix"],
    class_names=["SUPPORTS", "REFUTES", "NOT_ENOUGH_INFO"],
    output_path="../reports/baseline_confusion_matrix.png",
    title="FEVER Claim-Only Baseline — Confusion Matrix"
)